In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_RBF.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 2.0, 'n_it': 2.0}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=3)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}
            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.115658559129008

Trial 1 =========================================
15.132980937032308

Trial 2 =========================================
15.141117080485655

Trial 3 =========================================
15.143557004031646

Trial 4 =========================================
15.142241598656582

Trial 5 =========================================
15.142181110536384



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 6 =========================================
15.141889874670513

Trial 7 =========================================
15.140934094807156

Trial 8 =========================================
15.124095831177266

Trial 9 =========================================
15.134178206694433

Trial 10 =========================================
15.137069576122599

Trial 11 =========================================
15.142110671677809

Trial 12 =========================================
15.137835471269364

Trial 13 =========================================
15.123110259632579

Trial 14 =========================================
15.140800320579796

Trial 15 =========================================
15.124165609182201

Trial 16 =========================================
15.1278859892151

Trial 17 =========================================
15.14157310095323

Trial 18 =========================================
15.141508908557174

Trial 19 =========================================
15.141939254235291

Trial 20 ====

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 25 =========================================
15.132620983858093



...

Trial 88 =========================================
15.137273759289894

Trial 89 =========================================
15.12036757709962

Trial 90 =========================================
15.143324403632688

Trial 91 =========================================
15.128723531770056



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 92 =========================================
15.141223569400806

Trial 93 =========================================
15.134384833431985

Trial 94 =========================================
15.131309773077106



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 95 =========================================
15.136221467211504

Trial 96 =========================================
15.140166983074533

Trial 97 =========================================
15.11211808242667

Trial 98 =========================================
15.137486686456285

Trial 99 =========================================
15.14047545246654

[15.11565856 15.13298094 15.14111708 15.143557   15.1422416  15.14218111
 15.14188987 15.14093409 15.12409583 15.13417821 15.13706958 15.14211067
 15.13783547 15.12311026 15.14080032 15.12416561 15.12788599 15.1415731
 15.14150891 15.14193925 15.14286879 15.13946384 15.14046028 15.14206742
 15.13241646 15.13262098 15.1363644  15.13622674 15.14300573 15.140544
 15.14014782 15.14305736 15.14271534 15.13572554 15.14261092 15.13561781
 15.13735735 15.13412082 15.14261692 15.14106639 15.13315713 15.13794041
 15.12150823 15.13372303 15.14029442 15.14220581 15.12484549 15.13854914
 15.13648464 15.13987277 15.13656726 15.13622288 15.13709655 15.13

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 15.143557004031646
Avg = 15.136730764434683
Std = 0.006714891414882599


In [7]:
print(y_max_arr.tolist())

[15.115658559129008, 15.132980937032308, 15.141117080485655, 15.143557004031646, 15.142241598656582, 15.142181110536384, 15.141889874670513, 15.140934094807156, 15.124095831177266, 15.134178206694433, 15.137069576122599, 15.142110671677809, 15.137835471269364, 15.123110259632579, 15.140800320579796, 15.124165609182201, 15.1278859892151, 15.14157310095323, 15.141508908557174, 15.141939254235291, 15.14286879247029, 15.139463835025392, 15.14046027656955, 15.142067418045507, 15.132416461308068, 15.132620983858093, 15.136364401884492, 15.136226740550123, 15.143005725386285, 15.140544003490412, 15.140147817878628, 15.143057361773192, 15.142715341935007, 15.135725541859854, 15.142610924427538, 15.135617806985548, 15.137357345276195, 15.134120818000218, 15.142616919643238, 15.141066391557143, 15.133157130688105, 15.137940411432925, 15.121508233020878, 15.133723034910302, 15.140294424734101, 15.142205806524743, 15.124845486825645, 15.138549144151018, 15.136484640689625, 15.139872772891461, 15.1

In [8]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_RBF/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_RBF/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    15.139807
1    15.133235
2    15.138981
3    15.141145
4    15.110410
..         ...
995  15.136221
996  15.140167
997  15.112118
998  15.137487
999  15.140475

[1000 rows x 1 columns]
